# Gaussian Naive Bayes From Scratch

**Topic:** Probability and Statistics &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X_train` `(1500, 6)`, `y_train` `(1500,)`: 3 classes with **unequal priors** (roughly 50/30/20) and per-class feature means and variances.
- `X_test` `(500, 6)`: predict these. `y_test` is used by the evaluation cell.

## Your task

Implement Gaussian naive Bayes with **NumPy only** (no `sklearn.naive_bayes`):

1. Estimate per-class priors, per-class-per-feature means and variances from the training data.
2. Score each test point with `log prior + Σ_f log N(x_f | μ_cf, σ²_cf)`.
   The Gaussian log-density is `log N(x | μ, σ²) = -0.5 * log(2πσ²) - (x - μ)² / (2σ²)`.
3. Set `predictions` to the argmax class per test point.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

rng = np.random.default_rng(13)
d = 6
priors = np.array([0.5, 0.3, 0.2])
means = rng.normal(0, 2.2, (3, d))
stds = rng.uniform(0.8, 2.0, (3, d))

def make(n, seed):
    r = np.random.default_rng(seed)
    ys = r.choice(3, size=n, p=priors)
    X = means[ys] + r.normal(size=(n, d)) * stds[ys]
    return X, ys

X_train, y_train = make(1500, 101)
X_test, y_test = make(500, 202)
print(f"X_train {X_train.shape}, class counts {np.bincount(y_train)}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.85

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet, revisit your model and try again.")


## Solution

Estimate priors, per-class means, and per-class variances from the training split, then score every test point with `log prior + sum of per-feature Gaussian log-densities`, all vectorized with broadcasting. `argmax` over classes gives the prediction.

```python
import numpy as np

classes = np.unique(y_train)
logprior = np.log(np.array([(y_train == c).mean() for c in classes]))
mu = np.vstack([X_train[y_train == c].mean(axis=0) for c in classes])
var = np.vstack([X_train[y_train == c].var(axis=0) for c in classes])

# (n_test, n_classes, d) broadcast, summed over features
diff = X_test[:, None, :] - mu[None, :, :]
loglik = -0.5 * (np.log(2 * np.pi * var)[None] + diff**2 / var[None]).sum(axis=2)
predictions = classes[(logprior[None] + loglik).argmax(axis=1)]
```
